# Анализ данных и Статистика в Pandas

Этот ноутбук ведет нас от **базовых операций** (очистка, сортировка) к **продвинутому статистическому анализу**. 

Мы разберем математические формулы, стоящие за кодом, и научимся интерпретировать данные.

In [ ]:
import pandas as pd
import numpy as np

# Настройка: отображать числа с 2 знаками после запятой
pd.options.display.float_format = '{:,.2f}'.format

# 1. Создаем датасет (Пассажиры Титаника)
data = {
    'Name': ['Braund, Mr. Owen', 'Cumings, Mrs. John', 'Heikkinen, Miss. Laina', 'Futrelle, Mrs. Jacques', 'Allen, Mr. William', 'Moran, Mr. James', 'McCarthy, Mr. Timothy'],
    'Age': [22, 38, 26, 35, 35, np.nan, 54],  # Пропуск (NaN)
    'Fare': [7.25, 71.28, 7.92, 53.1, 8.05, 8.45, 51.86],
    'Class': [3, 1, 3, 1, 3, 3, 1],
    'Survived': [0, 1, 1, 1, 0, 0, 0]
}

df = pd.DataFrame(data)
df

---

## Блок 1. Базовые инструменты аналитика

Прежде чем нырять в формулы, нужно "пощупать" данные.

### 1.1 Быстрый взгляд: `info` и `describe`
Это первые команды, которые пишет аналитик.

In [ ]:
# Общая информация: типы данных и наличие пустых значений
print("--- Info ---")
df.info()

# Статистическая сводка (Count, Mean, Std, Min, Max)
print("\n--- Describe ---")
df.describe()

### 1.2 Работа с пропусками (Cleaning)
Пропуски искажают статистику. 
*   `isna()` — найти.
*   `fillna()` — заполнить (мягкий подход).
    *   *Важно:* Если данные имеют выбросы (очень богатые люди), лучше заполнять **медианой**, а не средним.

In [ ]:
# Сколько пропусков?
print("Пропуски:\n", df.isna().sum())

# Заполняем медианой (она устойчивее к выбросам, чем среднее)
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)

print(f"Пропуски заполнены значением: {median_age}")

### 1.3 Сортировка и Фильтрация

In [ ]:
# Сортировка по Цене
sorted_df = df.sort_values(by='Fare', ascending=False)

# nlargest — самый быстрый способ получить топ-N
top_3_rich = df.nlargest(3, 'Fare')
top_3_rich

---

## Блок 2. Feature Engineering (Создание признаков)

Как извлекать пользу из сырых данных.

### 2.1 Простые преобразования и Apply

In [ ]:
# Арифметика: Представим цены в тысячах (нормировка)
df['Fare_k'] = df['Fare'] / 1000

# Логика через функцию: Категоризация
def get_wealth_status(price):
    return 'Rich' if price > 50 else 'Normal'

df['Status'] = df['Fare'].apply(get_wealth_status)
df[['Name', 'Fare', 'Status']].head()

### 2.2 Продвинутая работа со строками (.str)
Pandas умеет работать с текстом векторно (быстро).

In [ ]:
# Извлечем Титул из имени (Mr, Miss, Mrs)
# Разделим по запятой, возьмем вторую часть, потом по точке
df['Title'] = df['Name'].str.split(',').str[1].str.split('.').str[0].str.strip()

print("Уникальные титулы:", df['Title'].unique())

---

## Блок 3. Глубокая Статистика и Математика

Теперь переходим к сути. Как описывать данные языком цифр.

### 3.1 Среднее (Mean) и Медиана (Median)
**Формула среднего:** $ \bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i $

*   **Mean:** Центр тяжести данных. Чувствительно к выбросам.
*   **Median:** Значение ровно посередине отсортированного списка (50-й перцентиль).

In [ ]:
mean_f = df['Fare'].mean()
median_f = df['Fare'].median()

print(f"Среднее: {mean_f:.2f}")
print(f"Медиана: {median_f:.2f}")

if mean_f > median_f:
    print("Вывод: Распределение скошено вправо (есть аномально богатые пассажиры).")

### 3.2 Дисперсия (Variance) и Стд. Отклонение (Std)
Показывают "кучность" стрельбы. Насколько данные разбросаны от среднего.

**Формула Дисперсии ($S^2$):**
$$ S^2 = \frac{\sum (x_i - \bar{x})^2}{n - 1} $$

**Формула Std ($S$ или $\sigma$):** Корень из дисперсии. Измеряется в тех же единицах, что и данные (доллары, годы).
$$ S = \sqrt{S^2} $$

In [ ]:
std_age = df['Age'].std()
print(f"Разброс возраста (std): {std_age:.2f} года")

# Интервал нормальности (Mean +/- Std)
print(f"Большинство пассажиров (68%) имеют возраст от {df['Age'].mean() - std_age:.0f} до {df['Age'].mean() + std_age:.0f} лет")

### 3.3 Коэффициент Вариации (CV)

**Проблема:** Стандартное отклонение (`std`) измеряется в тех же единицах, что и данные. 
- Std Возраста = 13 лет.
- Std Цены = 28 долларов.
*Что более вариативно? Сравнивать "годы" и "доллары" нельзя.*

**Решение:** Коэффициент вариации (CV). Это процентное отклонение.
$$ CV = \frac{\sigma (std)}{\mu (mean)} \times 100\% $$

Если $CV > 33\%$, данные считаются неоднородными.

In [ ]:

cv_fare = (df['Fare'].std() / df['Fare'].mean()) * 100
cv_age = (df['Age'].std() / df['Age'].mean()) * 100

print(f"Коэффициент вариации Цены: {cv_fare:.1f}%")
print(f"Коэффициент вариации Возраста: {cv_age:.1f}%")

if cv_fare > cv_age:
    print("Вывод: Цены на билеты 'скачут' гораздо сильнее, чем возраст пассажиров.")

### 3.4 Корреляция (Correlation)

Показывает линейную связь между переменными (Коэффициент Пирсона $r$).

**Как читать $r$:**
*   **1.0**: Идеальная прямая связь (растет А -> растет Б).
*   **0.0**: Связи нет (хаос).
*   **-1.0**: Обратная связь (растет А -> падает Б).

*Пример обратной связи: Чем выше класс каюты (число 1 меньше числа 3), тем выше цена.*

In [ ]:
# Считаем матрицу корреляций
corr_matrix = df[['Age', 'Fare', 'Survived', 'Class']].corr()

# Красивая визуализация цветом (чем краснее, тем сильнее связь)
corr_matrix.style.background_gradient(cmap='coolwarm', axis=None, vmin=-1, vmax=1)

### 3.5 Квантили (Quantiles) и IQR
Делят данные на части.
- `Q1` (25%): Четверть людей беднее этого значения.
- `Q3` (75%): Четверть людей богаче этого значения.
- `IQR` (Interquartile Range) = Q3 - Q1. Это диапазон, где живет "средний класс".

In [ ]:
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1

print(f"Типичный диапазон цен (IQR): {IQR:.2f}")

#### Межквартильный размах (IQR) - способ поиска аномалий
Самый надежный способ. Мы отсекаем хвосты распределения.

$$ НижняяГраница = Q1 - 1.5 \times IQR $$
$$ ВерхняяГраница = Q3 + 1.5 \times IQR $$

In [ ]:
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Все, что дороже {upper_bound:.2f}, считается выбросом по методу IQR.")

# Фильтруем, чтобы увидеть аномалии
outliers_iqr = df[(df['Fare'] < lower_bound) | (df['Fare'] > upper_bound)]
outliers_iqr[['Name', 'Fare', 'Class']]

### 3.6 Z-оценка (Z-Score) и Поиск аномалий
Переводит данные в "сигмы". Показывает, на сколько стандартных отклонений значение далеко от среднего.
$$ Z = \frac{x - \mu}{\sigma} $$
Обычно выбросом считается все, где модуль Z-score $> 3$ (но для маленьких выборок часто берут $> 2$ или $> 1.5$).

In [ ]:
df['Z_Fare'] = (df['Fare'] - df['Fare'].mean()) / df['Fare'].std()

# Ищем аномалии (кто отклонился больше чем на 1.5 сигмы)
anomalies = df[df['Z_Fare'] > 1.5]
print("Аномально дорогие билеты:")
anomalies[['Name', 'Fare', 'Z_Fare']]

### 3.7 Асимметрия (Skewness) и Эксцесс (Kurtosis)
- **Skewness:** Куда наклонен горб. 
    - `> 0`: Длинный хвост справа (как зарплаты: много бедных, мало олигархов).
- **Kurtosis:** Острота пика. 
    - Высокий куртозис = данные сгруппированы в центре, но есть "тяжелые хвосты" (риск внезапных выбросов).

In [ ]:
print(f"Асимметрия (Skew): {df['Fare'].skew():.2f}")
print(f"Эксцесс (Kurtosis): {df['Fare'].kurtosis():.2f}")

### 3.8 Стандартная ошибка среднего (SEM)
$$ SEM = \frac{\sigma}{\sqrt{n}} $$
Показывает точность нашего среднего. Если мы возьмем другую группу пассажиров, насколько сильно изменится среднее значение?
Чем меньше SEM, тем больше мы доверяем нашему среднему.

In [ ]:
sem = df['Age'].sem()
mean_val = df['Age'].mean()

print(f"Средний возраст: {mean_val:.1f} +/- {sem:.1f}")
print("Это доверительный интервал для среднего значения.")

---

## Блок 4. Агрегация и Сводные таблицы

Сравнение групп между собой.

In [ ]:
# Простая группировка
# Кто богаче: Выжившие или нет?
print(df.groupby('Survived')['Fare'].mean())

In [ ]:
# Pivot Table (Сводная таблица) — мощнейший инструмент
# Смотрим среднюю цену в разрезе Класса и Выживаемости

pivot = df.pivot_table(
    values='Fare', 
    index='Class', 
    columns='Survived',
    aggfunc=['mean', 'count'] # Считаем и среднее, и количество
)
pivot

### Вывод
Мы прошли путь от загрузки данных до расчета Z-оценок и доверительных интервалов. 

**Чек-лист аналитика:**
1.  `describe()` и `info()` — понять структуру.
2.  `isna()` -> `fillna()` — убрать мусор.
3.  `mean` vs `median` — проверить асимметрию.
4.  `std` и `CV` — понять риски и разброс.
5.  `groupby` / `pivot_table` — найти закономерности.